In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import when, col

#chargement de table

In [0]:
df = spark.table("project_cardio.gold.cardio_features")
display(df)

#Vérifie les colonne


In [0]:
print(df.columns)

In [0]:
df = df.withColumn(
    "gender_index",
    when(col("gender") == "male", 1)
    .when(col("gender") == "female", 2)
    .otherwise(0)
)

#Préparer les features

In [0]:
feature_cols = [
    "age_years",
    "gender_index",
    "height",
    "weight",
    "bmi",
    "tension_systolic",
    "tension_diastolic",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_ml = assembler.transform(df)

display(df_ml)

#Séparer train et test

In [0]:
train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)

print("Train :", train_data.count())
print("Test :", test_data.count())

#Entraîner le modèle

In [0]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="cardio"
)

model = lr.fit(train_data)

#Faire les prédictions


In [0]:
predictions = model.transform(test_data)

display(predictions.select("cardio", "prediction", "probability"))

#Évaluer le modèle

In [0]:
evaluator = BinaryClassificationEvaluator(
    labelCol="cardio",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(predictions)

print("AUC du modèle :", auc)

In [0]:
model.write().overwrite().save(
    "/Volumes/project_cardio/bronze/volum_card/cardio_model"
)